In [1]:
import pandas as pd

In [16]:
mapping = pd.read_parquet('results_557k/chunk_cluster_impacts_map_2026-03-11.parquet')
mapping.head()

,openalex_id,chunk_idx,cluster_id,Resource_pred,wellbeing_pred,Need_pred,PlanetaryBoundary_pred,Justice_pred
0,W1000416519,0,"[0, 1, 2]","[forests, freshwater, urban_land, wetlands]","[community, environment, housing, safety]",[shelter_and_living_conditions],"[biosphere_integrity, climate_change, freshwat...",[procedural]
1,W1000416519,1,"[3, 0, 4, 5, 6, 7, 8]","[marine_resources, wetlands]","[community, environment, housing, safety]",[shelter_and_living_conditions],"[biosphere_integrity, climate_change, land_sys...",[]
2,W1000499890,0,"[9, 10, 11, 12]",[fossil_fuels],[environment],"[communication_and_information, mobility, nutr...",[climate_change],[distributional]
3,W1000499890,1,"[13, 14, 15]","[biomass, fossil_fuels]","[environment, housing]","[communication_and_information, mobility, shel...",[climate_change],[transitional]
4,W1000905369,0,"[16, 17]",[],"[civic_engagement, community, education, healt...","[education, healthcare]",[],[recognitional]


In [17]:
geo = pd.read_parquet('results_557k/geo_taxonomy_chunks_2026-03-09.parquet')
geo.head()

,openalex_id,chunk_idx,regional_group,geographical_scopes,main_country_focus,error
0,W1000416519,0,North America,"[Countries, Local or subnational]","[United States, New Zealand]",NaN
1,W1000416519,1,North America,"[Local or subnational, Countries]",[United States],NaN
2,W1000499890,0,Asia-Pacific States,"[Countries, Local or subnational]","[Singapore, Italy]",NaN
3,W1000499890,1,World,[Countries],[],NaN
4,W1000757617,0,Western European,[Countries],[France],NaN


In [18]:
geo = geo[geo['error'].isna()].drop(columns=['error'])

In [19]:
total = mapping.merge(geo, on=['openalex_id', 'chunk_idx'], how='left')

In [20]:
clusters = pd.read_csv('impacts_analysis/clusters_with_sustainability_classification_temp_2026-03-13.csv')
clusters.head()

,cluster_id,text,count,sustainability_class
0,0,Insurance programs and tax incentives for floo...,715,PS
1,1,Preserving natural lands for flood management,447,S
2,2,Continuous monitoring of fishways to optimise ...,1536,S
3,3,Increasing restoration funding,79,S
4,4,Implementation of measures to mitigate the ris...,1031,NS


In [21]:
cluster_class_dict = clusters.set_index('cluster_id')['sustainability_class'].to_dict()

In [22]:
total['chunk_sufficiency_classes'] = total['cluster_id'].apply(lambda x: list(set(cluster_class_dict[i] for i in x)))

In [10]:
total.to_parquet('results_557k/gathered_results_by_chunk_2026-03-13.parquet')

## By publication

In [45]:
total = pd.read_parquet('results_557k/gathered_results_by_chunk_2026-03-13.parquet')

In [46]:
def gather(x):
    if x is None or isinstance(x, (int, float, str)):
        return [x]
    else:
         return list(x)

In [47]:
total[total.columns[1:]] = total[total.columns[1:]].map(gather)

In [54]:
publications = total.groupby('openalex_id').sum().map(set).map(list)
publications.head()

,chunk_idx,cluster_id,Resource_pred,wellbeing_pred,Need_pred,PlanetaryBoundary_pred,Justice_pred,regional_group,geographical_scopes,main_country_focus,chunk_sufficiency_classes
openalex_id,,,,,,,,,,,
W1000416519,"[0, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8]","[marine_resources, forests, wetlands, urban_la...","[housing, community, safety, environment]",[shelter_and_living_conditions],"[freshwater_use, climate_change, land_system_c...",[procedural],[North America],"[Local or subnational, Countries]","[United States, New Zealand]","[NR, S, NS, PS]"
W1000499890,"[0, 1]","[9, 10, 11, 12, 13, 14, 15]","[biomass, fossil_fuels]","[housing, environment]","[shelter_and_living_conditions, nutrition, mob...",[climate_change],"[distributional, transitional]","[World, Asia-Pacific States]","[Local or subnational, Countries]","[Italy, Singapore]","[S, NS, NR]"
W1000905369,"[0, 1]","[16, 17, 18, 19, 20, 21]",[],"[life_satisfaction, community, education, heal...","[healthcare, education]",[],[recognitional],"[Asia-Pacific States, North America]",[Countries],"[Turkey, Portugal, United States]","[NS, NR]"
W1001607710,[0],"[24, 22, 23]","[biomass, forests, agricultural_land]",[environment],[nutrition],"[land_system_change, biosphere_integrity]",[],[Eastern European States],[Local or subnational],[Serbia],[S]
W100238094,[0],"[25, 26, 27, 28]","[fossil_fuels, agricultural_land, biomass, met...","[housing, income, environment]","[shelter_and_living_conditions, nutrition, mob...",[climate_change],[transitional],[North America],[Countries],[United States],"[S, NS, PS]"


In [56]:
publications.to_parquet('results_557k/gathered_results_by_work_2026-03-13.parquet')

## Add OpenAlex metadata

In [2]:
publications = pd.read_parquet('results_557k/gathered_results_by_work_2026-03-13.parquet')

In [3]:
from itertools import chain
from typing import Generator
from pyalex import Works, Work
from tqdm.auto import tqdm

In [4]:
def get_works_from_ids(
    ids: list,
    fields: list[str],
    per_page: int = 100,
    filters: dict = {},
) -> Generator[Work, None, None]:
    """Fetch works given a list of OpenAlex IDs."""
    for i in range(0, len(ids), per_page):
        batch_ids = ids[i : i + per_page]
        works = (
            Works().filter(openalex_id="|".join(batch_ids)).filter(**filters).select(fields)
        )
        for work in chain(*works.paginate(per_page=per_page, n_max=None)):
            yield work

In [5]:
fields = [
    'id',                               # OpenAlex ID
    'title',                            # Titre
    'authorships',                      # Auteurs
    'type',                             # Type de document
    'primary_location',                 # Publication (journal)
    'publication_date',                 # Date
    'biblio',                           # Volume
    'doi',                              # DOI
    'ids',                              # ISBN (si pas de DOI)
]

In [6]:
ids = publications.index.tolist()
works = list(tqdm(get_works_from_ids(ids, fields), total=len(ids), desc="Fetching works"))

Fetching works:   0%|          | 0/226174 [00:00<?, ?it/s]

In [22]:
len(works)  # I had to relaunch for the last few IDs, but in the end we have everything

226174

In [23]:
meta = pd.json_normalize(works)
meta.head()

,id,title,authorships,type,publication_date,doi,primary_location.id,primary_location.is_oa,primary_location.landing_page_url,primary_location.pdf_url,...,primary_location.raw_type,biblio.volume,biblio.issue,biblio.first_page,biblio.last_page,ids.openalex,ids.mag,ids.doi,ids.pmid,primary_location.source
0,https://openalex.org/W1035580327,Climate Change and Agricultural Vulnerability,"[{'author_position': 'first', 'author': {'id':...",book,2002-01-01,NaN,pmh:oai:pure.iiasa.ac.at:6670,True,"https://pure.iiasa.ac.at/view/iiasa/89.html>,",http://pure.iiasa.ac.at/id/eprint/6670/1/XO-02...,...,NonPeerReviewed,NaN,NaN,NaN,NaN,https://openalex.org/W1035580327,1035580327,NaN,NaN,NaN
1,https://openalex.org/W101325984,Eutrophication assessment of coastal waters ba...,"[{'author_position': 'first', 'author': {'id':...",review,2013-04-30,https://doi.org/10.30955/gnj.000626,doi:10.30955/gnj.000626,True,https://doi.org/10.30955/gnj.000626,https://journal.gnest.org/sites/default/files/...,...,journal-article,11,4,373,390,https://openalex.org/W101325984,101325984,https://doi.org/10.30955/gnj.000626,NaN,NaN
2,https://openalex.org/W1023388751,Spatial Misfit in Participatory River Basin Ma...,"[{'author_position': 'first', 'author': {'id':...",article,2008-01-01,https://doi.org/10.5751/es-02341-130107,doi:10.5751/es-02341-130107,True,https://doi.org/10.5751/es-02341-130107,https://www.ecologyandsociety.org/vol13/iss1/a...,...,journal-article,13,1,NaN,NaN,https://openalex.org/W1023388751,1023388751,https://doi.org/10.5751/es-02341-130107,NaN,NaN
3,https://openalex.org/W1015139228,Design of production systems with hybrid energ...,"[{'author_position': 'first', 'author': {'id':...",article,2015-07-24,https://doi.org/10.1007/s10098-015-0947-4,doi:10.1007/s10098-015-0947-4,True,https://doi.org/10.1007/s10098-015-0947-4,https://link.springer.com/content/pdf/10.1007/...,...,journal-article,17,7,1807,1829,https://openalex.org/W1015139228,1015139228,https://doi.org/10.1007/s10098-015-0947-4,NaN,NaN
4,https://openalex.org/W1034120916,The Principles of Conservation and Development...,"[{'author_position': 'first', 'author': {'id':...",article,2007-01-01,https://doi.org/10.5751/es-02060-120202,doi:10.5751/es-02060-120202,True,https://doi.org/10.5751/es-02060-120202,http://www.ecologyandsociety.org/vol12/iss2/ar...,...,journal-article,12,2,NaN,NaN,https://openalex.org/W1034120916,1034120916,https://doi.org/10.5751/es-02060-120202,NaN,NaN


In [25]:
cols = ['id', 'title', 'authorships', 'type', 'publication_date', 'doi',
       'primary_location.landing_page_url', 'primary_location.pdf_url', 'primary_location.source.issn',
       'primary_location.source.host_organization_name',
       'primary_location.is_accepted', 'primary_location.is_published',
       'primary_location.raw_source_name', 'primary_location.raw_type',
       'biblio.volume', 'biblio.issue', 'biblio.first_page',
       'biblio.last_page', 'ids.openalex', 'ids.mag', 'ids.doi', 'ids.pmid',
       'primary_location.source']
meta = meta[cols]

In [26]:
def get_author_name(x):
    def extract_author_name(a):
        author_info = a.get('author', {})
        if 'raw_author_name' in author_info:
            return author_info['raw_author_name']
        return author_info.get('display_name', None)

    if isinstance(x, list):
        return [extract_author_name(a) for a in x]
    elif isinstance(x, dict):
        return extract_author_name(x)
    else:
        return None

In [27]:
meta['authorships'] = meta['authorships'].apply(get_author_name)

In [28]:
meta['openalex_id'] = meta['id'].apply(lambda x: x.split('/')[-1])

In [29]:
meta = meta.set_index('openalex_id')

In [31]:
final = publications.join(meta)

In [33]:
final.to_parquet('results_557k/gathered_results_by_work_with_openalex_metadata_2026-03-13.parquet')

In [36]:
final.reset_index().to_csv('results_557k/gathered_results_by_work_with_openalex_metadata_2026-03-13.csv', index=False)